# Thermal & Infrared Pedestrian Detection for Night Vision Surveillance
## Module A: 1-Channel YOLOv8 Model Training Pipeline in Google Colab

**Academic Project:** Thermal & Infrared Pedestrian Detection for Night Vision Surveillance  
**Target Object:** Pedestrian Detection (`class_id = 0`, person only)  
**Input Modality:** Genuine Single-Channel Thermal Infrared (LWIR 8-14 μm, Grayscale)  
**Network Backbone:** Adapted YOLOv8n (Input Layer 0: `Conv2d(1, 16, 3, 2, 1)`)  
**Pretrained Weight Strategy:** Channel-Averaged Weight Transplantation from COCO YOLOv8n  

---
### Instructions Before Running
1. **Enable GPU Hardware Acceleration:**
   - Navigate to `Runtime` → `Change runtime type`.
   - Select **T4 GPU** (or A100 / V100) → Click **Save**.
2. Execute each cell sequentially from Step 1 to Step 12.

### Step 1: GPU Environment Audit & Acceleration Verification
Audit CUDA availability, GPU device specifications, and VRAM capacity.

In [ ]:
!nvidia-smi

import torch
print("=" * 65)
print("HARDWARE ACCELERATION AUDIT")
print("=" * 65)
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Model       : {gpu_name}")
    print(f"Allocated VRAM  : {vram:.2f} GB")
    print(f"CUDA Capability : {torch.cuda.get_device_capability(0)}")
else:
    print("[!] WARNING: GPU not detected. Enable GPU acceleration under Runtime -> Change runtime type.")
print("=" * 65)

### Step 2: Install Dependencies
Install the latest Ultralytics package alongside OpenCV, PyYAML, Pillow, and Matplotlib.

In [ ]:
%pip install -q ultralytics opencv-python matplotlib pyyaml pillow tqdm

import ultralytics
import cv2
print(f"Ultralytics Version : {ultralytics.__version__}")
print(f"OpenCV Version      : {cv2.__version__}")

### Step 3: Project Directory Structure & Dataset Ingestion
Upload `processed.zip` (containing the 26,061 preprocessed single-channel PNG images and person-only labels) to `/content/` or mount your Google Drive.

In [ ]:
import os
import zipfile
from pathlib import Path

# Create project workspace
PROJECT_DIR = Path("/content/CV_Project")
(PROJECT_DIR / "configs").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "weights").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "data" / "processed").mkdir(parents=True, exist_ok=True)

# Search for uploaded processed dataset archive
zip_candidates = [
    "/content/processed.zip",
    "/content/data_processed.zip",
    "/content/drive/MyDrive/processed.zip"
]

extracted = False
for z in zip_candidates:
    if os.path.exists(z):
        print(f"Found dataset archive: {z}")
        print("Extracting into /content/CV_Project/data/processed ...")
        with zipfile.ZipFile(z, 'r') as zip_ref:
            zip_ref.extractall("/content/CV_Project/data/processed")
        extracted = True
        print("Extraction complete.")
        break

if not extracted:
    print("[!] NOTICE: Please upload 'processed.zip' to /content/ before proceeding.")
    print("    You can create this on your PC using:")
    print("    Compress-Archive -Path 'e:\\CV_Project\\data\\processed\\*' -DestinationPath 'e:\\CV_Project\\processed.zip'")

### Step 4: Verify Dataset Split Counts & Background Negatives
Audit all images, labels, and background negatives across splits.

#### Dataset Split & Background Negatives Table
| Split | Total Images | Positive Frames (With Pedestrians) | Background Negatives (Zero Annotations) | Pedestrian Annotations |
|:---|:---:|:---:|:---:|:---:|
| **Train** | 23,287 | 18,183 | **5,104** (3,257 missing labels + 1,847 empty labels) | 57,801 |
| **Validation** | 2,523 | 2,338 | **185** (0 missing labels + 185 empty labels) | 6,861 |
| **Test** | 251 | 237 | **14** (0 missing labels + 14 empty labels) | 777 |
| **TOTAL** | **26,061** | **20,758** | **5,303** | **65,439** |

> ### Critical Academic Note & Limitation Analysis: 3,257 Unlabelled Training Images
> In the FLIR 2024 source training split, **3,257 images** have no corresponding annotation `.txt` files.
> 
> **Potential Limitation & Label Noise:**
> - In real-world thermal surveillance datasets, unannotated frames often correspond to empty road sequences where no objects were observed. However, there is an inherent risk that some frames may contain faint, occluded, or distant pedestrians that human annotators missed.
> - **Impact on Training:** In YOLOv8, an unlabelled image is treated as a 100% negative background frame. If an unannotated pedestrian is present, the model's correct detection will be penalized by the loss function as a false positive, potentially reducing sensitivity to small/distant pedestrians.
> - **Engineering Tradeoff:** Retaining these 3,257 frames as hard background negatives teaches the model to suppress false alarms on heated vehicle engines, asphalt reflections, and warm building facades (a critical requirement for thermal surveillance). Retaining them without fabricating synthetic annotations preserves dataset integrity.

In [ ]:
import os
import glob
from pathlib import Path

data_root = PROJECT_DIR / "data" / "processed"
splits = ["train", "valid", "test"]

print("=" * 75)
print("DATASET INTEGRITY & BACKGROUND NEGATIVES AUDIT")
print("=" * 75)
total_imgs = 0
total_pos_imgs = 0
total_neg_imgs = 0
total_lbl_files = 0
total_persons = 0

expected_negatives = {"train": 5104, "valid": 185, "test": 14}

for sp in splits:
    img_files = sorted(glob.glob(str(data_root / sp / "images" / "*.png")))
    lbl_files = sorted(glob.glob(str(data_root / sp / "labels" / "*.txt")))
    lbl_set = {Path(f).stem: f for f in lbl_files}
    
    split_pos_imgs = 0
    split_neg_imgs = 0
    split_persons = 0
    
    for img_path in img_files:
        stem = Path(img_path).stem
        if stem not in lbl_set:
            # Missing label file = negative background frame
            split_neg_imgs += 1
        else:
            with open(lbl_set[stem], "r") as f:
                lines = [l.strip() for l in f if l.strip()]
            if len(lines) == 0:
                # Empty label file = negative background frame
                split_neg_imgs += 1
            else:
                split_pos_imgs += 1
                for line in lines:
                    assert line.startswith("0 "), f"Invalid class ID in {lbl_set[stem]}! Must be 0."
                    split_persons += 1
    
    total_imgs += len(img_files)
    total_lbl_files += len(lbl_files)
    total_pos_imgs += split_pos_imgs
    total_neg_imgs += split_neg_imgs
    total_persons += split_persons
    
    print(f"[{sp.upper():5s}] Images: {len(img_files):>6,} | Positive: {split_pos_imgs:>6,} | Negatives: {split_neg_imgs:>5,} | Pedestrians: {split_persons:>6,}")
    if len(img_files) > 0:
        assert split_neg_imgs == expected_negatives[sp], f"{sp} negative count mismatch: {split_neg_imgs} != {expected_negatives[sp]}"

print("-" * 75)
print(f"TOTAL  Images: {total_imgs:>6,} | Positive: {total_pos_imgs:>6,} | Negatives: {total_neg_imgs:>5,} | Pedestrians: {total_persons:>6,}")
print("=" * 75)
if total_imgs > 0:
    assert total_imgs == 26061, f"Expected 26,061 total images, got {total_imgs}"
    assert total_neg_imgs == 5303, f"Expected 5,303 total background negatives, got {total_neg_imgs}"
    assert total_pos_imgs == 20758, f"Expected 20,758 positive images, got {total_pos_imgs}"
    assert total_persons == 65439, f"Expected 65,439 pedestrian annotations, got {total_persons}"
    print("[PASS] All split image, background negative, and annotation counts match audit targets exactly.")


### Step 5: Configure `data_colab.yaml` with `channels: 1`
**The Single-Channel Dataloader Guarantee:**
Ultralytics' `YOLODataset` reads `channels = self.data.get('channels', 3)` and sets `self.cv2_flag = cv2.IMREAD_GRAYSCALE` if `channels: 1` is declared. This guarantees genuine single-channel loading without RGB triplication.

In [ ]:
import yaml

colab_data_cfg = {
    "path": str(data_root),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["person"],
    "channels": 1  # Crucial: forces cv2.IMREAD_GRAYSCALE in Ultralytics YOLODataset
}

colab_yaml_path = PROJECT_DIR / "configs" / "data_colab.yaml"
with open(colab_yaml_path, "w") as f:
    yaml.dump(colab_data_cfg, f, default_flow_style=False, sort_keys=False)

print(f"[+] Generated Colab Dataset Configuration: {colab_yaml_path}")
print(open(colab_yaml_path).read())

### Step 6: Verify Dataloader Yields Genuine 1-Channel Tensors
Directly inspect an item from `YOLODataset` to confirm `cv2_flag == 0` (`IMREAD_GRAYSCALE`) and tensor shape is `[1, 640, 640]`.

In [ ]:
from ultralytics.data.dataset import YOLODataset
from ultralytics.cfg import get_cfg

cfg = get_cfg()
sample_ds = YOLODataset(
    img_path=str(data_root / "valid" / "images"),
    imgsz=640,
    batch_size=1,
    augment=False,
    hyp=cfg,
    data=colab_data_cfg,
    fraction=0.005  # audit sample
)

print(f"Dataloader channels attribute : {sample_ds.channels}")
print(f"Dataloader cv2_flag (0=GRAY)  : {sample_ds.cv2_flag}")
assert sample_ds.channels == 1, "Error: Dataloader channels != 1!"
assert sample_ds.cv2_flag == 0, "Error: cv2_flag != cv2.IMREAD_GRAYSCALE!"

sample_item = sample_ds[0]
img_tensor = sample_item['img']
print(f"Loaded Batch Image Shape      : {img_tensor.shape} (Channel dim=1)")
print(f"Loaded Batch Class Tensor     : {sample_item['cls']}")
assert img_tensor.shape[0] == 1, f"Expected 1 channel, got {img_tensor.shape[0]}"
print("[PASS] Ultralytics dataloader verified: 100% genuine single-channel infrared input.")

### Step 7: Load 1-Channel Checkpoint or Transplant Weights
Loads the verified Phase 3 checkpoint `weights/yolov8n_thermal_1ch.pt` (or transplants weights via channel-averaging from `yolov8n.pt` if starting fresh).

In [ ]:
from ultralytics import YOLO
from ultralytics.nn.tasks import DetectionModel

weights_dest = PROJECT_DIR / "weights" / "yolov8n_thermal_1ch.pt"

# Create 1-Channel Architecture YAML
arch_1ch_yaml = """
nc: 1
scales:
  n: [0.33, 0.25, 1024]
ch: 1

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [[15, 18, 21], 1, Detect, [nc]]
"""
arch_path = PROJECT_DIR / "configs" / "yolov8n_1ch.yaml"
with open(arch_path, "w") as f:
    f.write(arch_1ch_yaml)

if weights_dest.exists():
    print(f"[+] Loading existing Phase 3 verified checkpoint: {weights_dest}")
    thermal_model = YOLO(str(weights_dest))
else:
    print(f"[!] Checkpoint not found at {weights_dest}. Running on-the-fly weight transplantation...")
    source_yolo = YOLO("yolov8n.pt")
    source_dict = source_yolo.model.state_dict()
    
    target_model = DetectionModel(cfg=str(arch_path), ch=1, nc=1)
    target_dict = target_model.state_dict()
    target_model.nc = 1
    target_model.names = {0: "person"}
    
    new_state = {}
    for k, tgt_param in target_dict.items():
        if k in source_dict:
            src_param = source_dict[k]
            if src_param.shape == tgt_param.shape:
                new_state[k] = src_param.clone()
            elif k == "model.0.conv.weight":
                # Channel-average W_1ch = mean(W_3ch, dim=1)
                new_state[k] = src_param.mean(dim=1, keepdim=True)
            elif "cv3" in k:
                slices = tuple(slice(0, min(s, t)) for s, t in zip(src_param.shape, tgt_param.shape))
                init_val = tgt_param.clone()
                init_val[slices] = src_param[slices]
                new_state[k] = init_val
            else:
                new_state[k] = tgt_param
        else:
            new_state[k] = tgt_param
            
    target_model.load_state_dict(new_state)
    torch.save({
        "model": target_model,
        "yaml": str(arch_path),
        "epoch": -1,
        "train_args": {"ch": 1, "nc": 1},
        "names": {0: "person"}
    }, str(weights_dest))
    thermal_model = YOLO(str(weights_dest))
    print(f"[+] Created and saved 1-channel checkpoint: {weights_dest}")

# Confirm first layer conv in_channels
first_conv = thermal_model.model.model[0].conv
print(f"Layer 0 Conv verified: {first_conv}")
assert first_conv.in_channels == 1, "Layer 0 in_channels != 1!"

### Step 8: Input/Output Anchor Consistency Audit

**Mathematical Anchor Derivation:**
- **Native FLIR 4:3 Aspect Ratio (480 × 640):**
  $$\text{P3: } \frac{480}{8} \times \frac{640}{8} = 4,800 \quad | \quad \text{P4: } \frac{480}{16} \times \frac{640}{16} = 1,200 \quad | \quad \text{P5: } \frac{480}{32} \times \frac{640}{32} = 300$$
  $$\text{Total Anchors} = 4,800 + 1,200 + 300 = \mathbf{6,300} \quad \implies \quad \text{Output Shape: } [B, 5, 6300]$$

- **Standard Square Training Grid (640 × 640):**
  $$\text{P3: } \frac{640}{8} \times \frac{640}{8} = 6,400 \quad | \quad \text{P4: } \frac{640}{16} \times \frac{640}{16} = 1,600 \quad | \quad \text{P5: } \frac{640}{32} \times \frac{640}{32} = 400$$
  $$\text{Total Anchors} = 6,400 + 1,600 + 400 = \mathbf{8,400} \quad \implies \quad \text{Output Shape: } [B, 5, 8400]$$

The cell below verifies both mathematical properties:

In [ ]:
thermal_model.model.eval()

# 1. Test Native FLIR 480x640 Input
x_480_640 = torch.randn(1, 1, 480, 640)
with torch.no_grad():
    out_480_640 = thermal_model.model(x_480_640)
out_tensor_480_640 = out_480_640[0] if isinstance(out_480_640, (list, tuple)) else out_480_640

print(f"Input [1, 1, 480, 640] -> Output Shape : {list(out_tensor_480_640.shape)}")
assert out_tensor_480_640.shape == (1, 5, 6300), f"Expected (1, 5, 6300), got {out_tensor_480_640.shape}"
print("  [PASS] Native 480x640 input correctly produces 6,300 anchor candidates.")

# 2. Test Square 640x640 Input
x_640_640 = torch.randn(1, 1, 640, 640)
with torch.no_grad():
    out_640_640 = thermal_model.model(x_640_640)
out_tensor_640_640 = out_640_640[0] if isinstance(out_640_640, (list, tuple)) else out_640_640

print(f"Input [1, 1, 640, 640] -> Output Shape : {list(out_tensor_640_640.shape)}")
assert out_tensor_640_640.shape == (1, 5, 8400), f"Expected (1, 5, 8400), got {out_tensor_640_640.shape}"
print("  [PASS] Square 640x640 input correctly produces 8,400 anchor candidates.")

### Step 9: Train 1-Channel Thermal YOLOv8 on Colab GPU

#### Hyperparameters Tailored for Thermal Infrared
- `data="configs/data_colab.yaml"` (Contains `channels: 1`)
- `epochs=50`, `batch=16`, `imgsz=640`
- `hsv_h=0.0`, `hsv_s=0.0` (Hue/saturation disabled for single-channel thermal)
- `hsv_v=0.2` (Value/brightness jitter for infrared dynamic range variation)
- `fliplr=0.5` (Horizontal flip)
- `mosaic=1.0`, `close_mosaic=10`
- **Checkpointing:** Automatically saves `best.pt` and `last.pt` to `runs/detect/train_thermal_1ch/weights/`

In [ ]:
# Execute training session
train_results = thermal_model.train(
    data=str(colab_yaml_path),
    epochs=50,
    batch=16,
    imgsz=640,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/content/CV_Project/runs/detect",
    name="train_thermal_1ch",
    save=True,
    save_period=5,
    patience=15,
    plots=True,
    val=True,
    # Thermal-specific augmentations
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.2,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0,
    close_mosaic=10,
    optimizer="auto",
    verbose=True
)

# Verify best.pt and last.pt checkpoints exist
run_dir = Path("/content/CV_Project/runs/detect/train_thermal_1ch")
best_pt = run_dir / "weights" / "best.pt"
last_pt = run_dir / "weights" / "last.pt"

assert best_pt.exists(), f"Error: best.pt not found at {best_pt}"
assert last_pt.exists(), f"Error: last.pt not found at {last_pt}"
print(f"\n[PASS] Successfully trained and saved checkpoints:")
print(f"  best.pt : {best_pt} ({best_pt.stat().st_size / 1024**2:.2f} MB)")
print(f"  last.pt : {last_pt} ({last_pt.stat().st_size / 1024**2:.2f} MB)")

### Step 10: Training Curves, Losses & Confusion Matrix
Display training plots and metric progression curves generated during training.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

plot_files = ["results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"]
for pf in plot_files:
    p = run_dir / pf
    if p.exists():
        plt.figure(figsize=(10, 6))
        img = mpimg.imread(str(p))
        plt.imshow(img)
        plt.title(pf, fontsize=14)
        plt.axis("off")
        plt.show()

### Step 11: Quantitative Test Split Evaluation
**Strict Isolation:** Evaluate the trained `best.pt` exclusively on the 251 unseen test images (`test/images/`). The test set is never used during training or checkpoint selection.

In [ ]:
eval_model = YOLO(str(best_pt))

test_metrics = eval_model.val(
    data=str(colab_yaml_path),
    split="test",
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else "cpu"
)

print("=" * 65)
print("TEST SPLIT QUANTITATIVE METRICS (251 TEST FRAMES)")
print("=" * 65)
print(f"Precision (P)       : {test_metrics.box.p[0]:.4f}")
print(f"Recall (R)          : {test_metrics.box.r[0]:.4f}")
print(f"mAP@0.50            : {test_metrics.box.map50:.4f}")
print(f"mAP@0.50:0.95       : {test_metrics.box.map:.4f}")
print(f"Inference Latency   : {test_metrics.speed['inference']:.2f} ms/frame")
print("=" * 65)

### Step 12: Qualitative Thermal Visualizer & Download Package
Run pedestrian detection on sample test frames and download the complete results archive.

In [ ]:
import shutil
from google.colab import files

# 1. Visualize Sample Detections
test_samples = sorted(glob.glob(str(data_root / "test" / "images" / "*.png")))[:4]
if test_samples:
    preds = eval_model.predict(test_samples, conf=0.25, save=False)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for idx, (res, ax) in enumerate(zip(preds, axes.ravel())):
        annotated_bgr = res.plot()
        annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
        ax.imshow(annotated_rgb)
        ax.set_title(f"Test Frame {idx+1}: {len(res.boxes)} Pedestrians Detected")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

# 2. Archive Checkpoints and Metrics for Download
archive_dest = "/content/thermal_yolov8_training_results"
shutil.make_archive(archive_dest, "zip", str(run_dir))
print(f"Created: {archive_dest}.zip")
files.download(f"{archive_dest}.zip")